In [13]:
!pip install -q langchain-text-splitters

In [1]:
!pip install -q langchain langchain-openai langchain-community chromadb openai tiktoken pypdf fastapi uvicorn nest-asyncio pyngrok python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.0/346.0 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.

In [4]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
print("Key set:", os.environ["OPENAI_API_KEY"][:7] + "...")

Enter your OpenAI API key: ··········
Key set: sk-proj...


In [6]:
import os


folders = [
    "document-intelligence-rag",
    "document-intelligence-rag/src",
    "document-intelligence-rag/data",
    "document-intelligence-rag/notebook",
]
for f in folders:
    os.makedirs(f, exist_ok=True)

print("Folders created ")
!ls -R document-intelligence-rag

Folders created 
document-intelligence-rag:
data  notebook	src

document-intelligence-rag/data:

document-intelligence-rag/notebook:

document-intelligence-rag/src:


In [8]:
%%writefile document-intelligence-rag/src/config.py
"""Central configuration for the Document Intelligence RAG system."""


CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150


EMBEDDING_MODEL = "text-embedding-3-small"
LLM_MODEL = "gpt-4o-mini"
LLM_TEMPERATURE = 0.0

CHROMA_DIR = "chroma_store"

TOP_K = 4

Overwriting document-intelligence-rag/src/config.py


In [10]:
%%writefile document-intelligence-rag/src/__init__.py
# Marks 'src' as a Python package

Writing document-intelligence-rag/src/__init__.py


In [14]:
%%writefile document-intelligence-rag/src/ingest.py
"""Load documents (PDF + text) and split them into chunks."""

import os
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from src.config import CHUNK_SIZE, CHUNK_OVERLAP


def load_document(file_path: str):
    """Load a single file. Supports .pdf and .txt."""
    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".pdf":
        loader = PyPDFLoader(file_path)
    elif ext == ".txt":
        loader = TextLoader(file_path, encoding="utf-8")
    else:
        raise ValueError(f"Unsupported file type: {ext}. Use .pdf or .txt")

    return loader.load()


def chunk_documents(documents):
    """Split documents into overlapping chunks for context optimization."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    return splitter.split_documents(documents)


def ingest_file(file_path: str):
    """Full pipeline for one file: load -> chunk. Returns list of chunks."""
    docs = load_document(file_path)
    chunks = chunk_documents(docs)
    print(f"Loaded '{file_path}' -> {len(docs)} page(s) -> {len(chunks)} chunk(s)")
    return chunks

Overwriting document-intelligence-rag/src/ingest.py


In [15]:
import importlib
import src.ingest
importlib.reload(src.ingest)
from src.ingest import ingest_file

chunks = ingest_file("document-intelligence-rag/data/sample.txt")
print("\nFirst chunk preview:")
print(chunks[0].page_content[:200])

Loaded 'document-intelligence-rag/data/sample.txt' -> 1 page(s) -> 1 chunk(s)

First chunk preview:
Retrieval Augmented Generation (RAG) combines a retriever and a language model.
The retriever finds relevant document chunks from a vector database.
The language model then generates an answer grounde


In [16]:
%%writefile document-intelligence-rag/src/vectorstore.py
"""ChromaDB vector store + OpenAI embeddings."""

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

from src.config import EMBEDDING_MODEL, CHROMA_DIR, TOP_K


def get_embeddings():
    """Create the OpenAI embedding function."""
    return OpenAIEmbeddings(model=EMBEDDING_MODEL)


def build_vectorstore(chunks):
    """Embed chunks and store them in ChromaDB. Returns the vector store."""
    embeddings = get_embeddings()
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=CHROMA_DIR,
    )
    print(f"Stored {len(chunks)} chunk(s) in ChromaDB at '{CHROMA_DIR}'")
    return vectorstore


def load_vectorstore():
    """Load an existing ChromaDB store from disk."""
    embeddings = get_embeddings()
    return Chroma(
        persist_directory=CHROMA_DIR,
        embedding_function=embeddings,
    )


def get_retriever(vectorstore):
    """Turn the vector store into a retriever that fetches top-K relevant chunks."""
    return vectorstore.as_retriever(search_kwargs={"k": TOP_K})

Writing document-intelligence-rag/src/vectorstore.py


In [17]:
import importlib
import src.vectorstore
importlib.reload(src.vectorstore)
from src.vectorstore import build_vectorstore, get_retriever

# Build the store from the chunks we made earlier
vectorstore = build_vectorstore(chunks)

# Turn it into a retriever and test a query
retriever = get_retriever(vectorstore)
results = retriever.invoke("What does the retriever do?")

print("\nTop retrieved chunk:")
print(results[0].page_content)

Stored 1 chunk(s) in ChromaDB at 'chroma_store'

Top retrieved chunk:
Retrieval Augmented Generation (RAG) combines a retriever and a language model.
The retriever finds relevant document chunks from a vector database.
The language model then generates an answer grounded in those chunks.
This reduces hallucination because answers are based on real retrieved context.
ChromaDB is a popular open-source vector database used to store embeddings.


In [18]:
%%writefile document-intelligence-rag/src/rag_chain.py
"""RAG chain: retrieve context + generate a grounded answer with OpenAI GPT."""

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from src.config import LLM_MODEL, LLM_TEMPERATURE


# Prompt that forces the model to answer ONLY from retrieved context.
# This is the "reduce hallucination" part: if the answer isn't in the
# context, the model is told to say so instead of making something up.
PROMPT_TEMPLATE = """You are a helpful assistant answering questions about documents.
Use ONLY the context below to answer the question.
If the answer is not in the context, say "I don't have enough information to answer that."

Context:
{context}

Question: {question}

Answer:"""


def format_docs(docs):
    """Join retrieved chunks into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)


def build_rag_chain(retriever):
    """Build the full RAG chain: retrieve -> prompt -> GPT -> text answer."""
    llm = ChatOpenAI(model=LLM_MODEL, temperature=LLM_TEMPERATURE)
    prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)

    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain


def answer_question(chain, question: str):
    """Run a question through the RAG chain and return the answer."""
    return chain.invoke(question)

Writing document-intelligence-rag/src/rag_chain.py


In [19]:
import importlib
import src.rag_chain
importlib.reload(src.rag_chain)
from src.rag_chain import build_rag_chain, answer_question

chain = build_rag_chain(retriever)

# Ask a question that the document can answer
q1 = "What is RAG and how does it reduce hallucination?"
print("Q:", q1)
print("A:", answer_question(chain, q1))

# Ask one the document can't answer (should say it doesn't know)
q2 = "Who won the 2022 World Cup?"
print("\nQ:", q2)
print("A:", answer_question(chain, q2))

Q: What is RAG and how does it reduce hallucination?
A: Retrieval Augmented Generation (RAG) combines a retriever and a language model. The retriever finds relevant document chunks from a vector database, and the language model generates an answer grounded in those chunks. This approach reduces hallucination because the answers are based on real retrieved context.

Q: Who won the 2022 World Cup?
A: I don't have enough information to answer that.


In [21]:
from getpass import getpass
from pyngrok import ngrok, conf

ngrok_token = getpass("Enter your ngrok authtoken: ")
conf.get_default().auth_token = ngrok_token
print("ngrok token set ")

Enter your ngrok authtoken: ··········
ngrok token set 


In [25]:
%%writefile document-intelligence-rag/src/api.py
"""FastAPI app exposing document ingestion and query endpoints."""

import os
import shutil
from fastapi import FastAPI, UploadFile, File
from pydantic import BaseModel

from src.ingest import ingest_file
from src.vectorstore import build_vectorstore, load_vectorstore, get_retriever
from src.rag_chain import build_rag_chain, answer_question
from src.config import CHROMA_DIR

app = FastAPI(title="Document Intelligence RAG API")

# Holds the active RAG chain in memory after documents are ingested
state = {"chain": None}

UPLOAD_DIR = "uploads"
os.makedirs(UPLOAD_DIR, exist_ok=True)


class QueryRequest(BaseModel):
    """Request body for the /query endpoint."""
    question: str


@app.get("/")
def health():
    """Simple health check."""
    return {"status": "ok", "message": "Document Intelligence RAG API is running"}


@app.post("/ingest")
async def ingest(file: UploadFile = File(...)):
    """Upload a PDF or TXT file, chunk it, and store it in ChromaDB."""
    file_path = os.path.join(UPLOAD_DIR, file.filename)
    with open(file_path, "wb") as f:
        shutil.copyfileobj(file.file, f)

    chunks = ingest_file(file_path)
    vectorstore = build_vectorstore(chunks)
    retriever = get_retriever(vectorstore)
    state["chain"] = build_rag_chain(retriever)

    return {
        "filename": file.filename,
        "chunks_stored": len(chunks),
        "message": "Document ingested. You can now query it.",
    }


@app.post("/query")
def query(request: QueryRequest):
    """Ask a question against the ingested documents."""
    if state["chain"] is None:
        if os.path.exists(CHROMA_DIR):
            retriever = get_retriever(load_vectorstore())
            state["chain"] = build_rag_chain(retriever)
        else:
            return {"error": "No documents ingested yet. Use /ingest first."}

    answer = answer_question(state["chain"], request.question)
    return {"question": request.question, "answer": answer}

Writing document-intelligence-rag/src/api.py


In [27]:
import nest_asyncio
import uvicorn
import threading
from pyngrok import ngrok

import importlib
import src.api
importlib.reload(src.api)
from src.api import app

# Allow uvicorn to run inside Colab's existing event loop
nest_asyncio.apply()

# Kill any old tunnels/servers from previous runs
ngrok.kill()

# Open a public URL pointing to port 8000
tunnel = ngrok.connect(8000)
public_url = tunnel.public_url
print(" Public API URL:", public_url)
print(" Interactive docs:", public_url + "/docs")

# Run uvicorn in a background thread so the notebook stays responsive
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="error")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
print("Server starting... give it ~3 seconds.")

 Public API URL: https://scurvy-ripeness-humped.ngrok-free.dev
 Interactive docs: https://scurvy-ripeness-humped.ngrok-free.dev/docs
Server starting... give it ~3 seconds.


In [28]:
%%writefile document-intelligence-rag/requirements.txt
langchain
langchain-openai
langchain-community
langchain-text-splitters
chromadb
openai
tiktoken
pypdf
fastapi
uvicorn
nest-asyncio
pyngrok
python-dotenv
pydantic

Writing document-intelligence-rag/requirements.txt


In [29]:
%%writefile document-intelligence-rag/.gitignore
# Secrets - NEVER commit these
.env
*.key

# Python
__pycache__/
*.pyc
.ipynb_checkpoints/

# Vector store and uploads (generated at runtime)
chroma_store/
uploads/

# OS
.DS_Store

Writing document-intelligence-rag/.gitignore


In [30]:
%%writefile document-intelligence-rag/.env.example
# Copy this file to .env and fill in your real key
OPENAI_API_KEY=your-openai-api-key-here

Writing document-intelligence-rag/.env.example


In [31]:
%%writefile document-intelligence-rag/README.md
# Document Intelligence System using RAG and LLMs

A Retrieval Augmented Generation (RAG) pipeline that enables semantic
question-answering over custom document collections (PDF and text). Built with
LangChain and OpenAI GPT, using ChromaDB for vector storage and configurable
chunking for context optimization. Document ingestion and query endpoints are
exposed via FastAPI, with responses grounded in retrieved context to reduce
hallucination on domain-specific queries.

## Features

- Semantic Q&A over custom PDF and text documents
- LangChain + OpenAI GPT RAG pipeline
- ChromaDB vector storage with OpenAI embeddings
- Configurable chunking (chunk size + overlap) for context optimization
- FastAPI endpoints for document ingestion and querying
- Grounded responses that reduce hallucination by answering only from retrieved context

## Tech Stack

Python, LangChain, OpenAI API, ChromaDB, FastAPI, Retrieval Augmented Generation (RAG)

## Architecture

Writing document-intelligence-rag/README.md


In [40]:
from getpass import getpass

github_token = getpass("Enter your GitHub token (ghp_...): ")
github_username = "HarshaLolabattu"
repo_name = "document-intelligence-rag"

# Configure git identity
!git config --global user.name "Harsha Adinaraynaraju"
!git config --global user.email "harshaadinarayana@gmail.com"

print("Token captured and git configured ")

Enter your GitHub token (ghp_...): ··········
Token captured and git configured 


In [41]:
import os
os.chdir("/content/document-intelligence-rag")

# Initialize git and set the main branch
!git init -q
!git branch -M main

# Add the remote using the token for authentication
remote_url = f"https://{github_token}@github.com/{github_username}/{repo_name}.git"
!git remote remove origin 2>/dev/null
!git remote add origin "{remote_url}"

# Stage everything (the .gitignore protects secrets + chroma_store + uploads)
!git add .
!git commit -q -m "Initial commit: Document Intelligence RAG system"

# Push to GitHub
!git push -u origin main --force

Enumerating objects: 15, done.
Counting objects: 100% (15/15), done.
Delta compression using up to 2 threads
Compressing objects: 100% (13/13), done.
Writing objects: 100% (15/15), 4.50 KiB | 2.25 MiB/s, done.
Total 15 (delta 0), reused 0 (delta 0), pack-reused 0
To https://github.com/HarshaLolabattu/document-intelligence-rag.git
 * [new branch]      main -> main
Branch 'main' set up to track remote branch 'main' from 'origin'.
